# Rules vs. Learned Detectors — San Diego "Get It Done" 311 data quality
CS 677 final project · Devashish Palate, Mohammadreza Ghassemzadeh

Runs the whole pipeline on Colab: profiling → rule tests → duplicate detection (RQ2) → error-injection study (RQ1) → cleaning and district analysis (RQ3) → figures. **Runtime → Run all.** Full run ≈ 20 min; set `QUICK = True` below for a ~3 min test.

## 1. Get the code
If you opened this notebook from GitHub, set `REPO_URL` to your repo. Otherwise leave it empty and upload `CS677_project_code.zip` when asked.

In [ ]:
REPO_URL = "https://github.com/<your-username>/cs677-get-it-done-dq.git"  # <- edit
QUICK = False   # True: 1 seed, 10k sample for the injection study

import os
if not os.path.exists("/content/project"):
    if "<your-username>" not in REPO_URL:
        !git clone -q {REPO_URL} /content/project
    else:
        from google.colab import files
        up = files.upload()            # choose the project zip
        z = next(iter(up))
        !mkdir -p /content/project && unzip -q -o "{z}" -d /content/project
%cd /content/project
!ls

In [ ]:
!pip -q install rapidfuzz

## 2. Get the data (378,669 rows, 128 MB)
Downloads directly from the City of San Diego open data portal. If the URL ever changes, download the CSV from data.sandiego.gov and upload it to `data/gid2025.csv`.

In [ ]:
!python src/get_data.py

## 3. Profile + rule suite

In [ ]:
!python src/run_profile.py

In [ ]:
!python -m pytest tests -q -p no:cacheprovider   # 28 unit tests pass; 8 data assertions fail = documented defects

## 4. RQ2 — duplicate detection vs. City labels

In [ ]:
!python src/run_duplicates.py

## 5. RQ1 — error-injection study

In [ ]:
if QUICK:
    !N_SAMPLE=10000 SEEDS=1 python src/run_injection.py
else:
    !python src/run_injection.py

## 6. RQ3 — cleaning and district response times

In [ ]:
!python src/run_cleaning.py

## 7. Figures and tables

In [ ]:
!python src/make_figs.py && python src/make_tables.py
from IPython.display import Image, display
for f in ["injection_heatmap", "ml_avg_precision", "dup_sweep", "district_response", "map_flags"]:
    print(f); display(Image(f"figs/{f}.png", width=900))

## 8. Download results

In [ ]:
!zip -q -r results_and_figs.zip results figs report/injection_table.tex
from google.colab import files
files.download("results_and_figs.zip")